In [2]:
import torch
import torch.optim as optim
from transformers import T5ForConditionalGeneration, T5Tokenizer
import time
from typing import List, Tuple
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset

Skipping import of cpp extensions due to incompatible torch version 2.7.1+cu118 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1220 16:12:03.999000 8944 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

model_name = "google/flan-t5-base"
print(f"Loading model: {model_name}...")
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

print(f"Model loaded: {model_name}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size:,}")
print()

Using device: cuda
GPU: NVIDIA GeForce RTX 3080
Memory allocated: 0.00 GB
Loading model: google/flan-t5-base...


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Model loaded: google/flan-t5-base
Model parameters: 247,577,856
Tokenizer vocab size: 32,000



In [7]:

def generate_responses(message, n=3):
    prompt = f"Write a reply in your normal texting style: {message}"
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        num_return_sequences=n,
        do_sample=True,
        temperature=0.3,  # Increased from 0.8 (more randomness)
        top_p=0.9,       # Nucleus sampling for diversity
        top_k=50,        # Limit to top 50 tokens
        repetition_penalty=1.2,  # Lower penalty = more generic
        num_beams=5,     # Add beams for better quality
    )
    
    return [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]


In [8]:
# Cell 9: Test Model
test_messages = [
    "Hey, what's up?",
    "How are you doing?",
    "See you tomorrow",
    "have you finished the report?"
]

print("Testing model...")
for msg in test_messages:
    response = generate_responses(msg)
    print(f"Input: {msg}")
    print(f"Response: {response}")
    print("-" * 40)

Testing model...
Input: Hey, what's up?
Response: ["Hey, what's up?", 'Hey, what are you up to?', 'Hey, what are you doing?']
----------------------------------------
Input: How are you doing?
Response: ["I'm fine.", "I'm fine. I haven't seen you in a while.", "I'm fine. I haven't seen you for a while."]
----------------------------------------
Input: See you tomorrow
Response: ["I'll see you tomorrow.", "I can't wait to see you tomorrow!", "I can't wait to see you tomorrow."]
----------------------------------------
Input: have you finished the report?
Response: ["I haven't finished the report yet.", "I haven't done the report yet.", "I haven't finished it yet."]
----------------------------------------
